# Rice Disease Classification - DenseNet121 From-Scratch PyTorch Pipeline

Notebook này thay thế bản Colab/TensorFlow cũ bằng một pipeline PyTorch tự viết theo cấu trúc tham khảo từ `Lab_5_1_ViT_template.ipynb`:

1. Cài/kiểm tra thư viện và GPU
2. Cấu hình đường dẫn, class names, seed
3. Dataset/DataLoader tự viết
4. Visualize batch mẫu
5. Build DenseNet121 model
6. Training loop tự viết
7. Fine-tune 2 pha
8. Vẽ learning curves
9. Final evaluation + confusion matrix
10. Inference demo + tạo `submission.csv`

Thiết kế hiện tại là **image-only**. Notebook không dùng `field_id`, `province`, `season`, `variety`, `age`, hoặc metadata giả vì dataset hiện tại không có metadata hiện trường đáng tin cậy.


## 1. Cài đặt và import thư viện

Nếu chạy trong môi trường mới, cài PyTorch CUDA trước khi mở notebook:

```powershell
uv run --python 3.11 --index https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --with "torch==2.11.0+cu128" --with "torchvision==0.26.0+cu128" --with pandas --with numpy --with pillow --with scikit-learn python scripts/run_densenet_gpu_image_only_fixed.py
```

Trong notebook, chỉ chạy cell cài đặt nếu thiếu package.


In [ ]:
# Nếu thiếu thư viện trong notebook runtime, bỏ comment dòng dưới.
# !pip install torch torchvision pandas numpy pillow scikit-learn matplotlib seaborn tqdm

from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


## 2. Cấu hình thí nghiệm

Notebook mặc định dùng manifest đã sửa lỗi:

- `reports/suggested_runs/image_only_fix/image_only_hash_split_manifest.csv`
- Split stratified theo label và group theo `dhash`
- Không dùng metadata giả
- Không dùng class weights mặc định vì run trước cho thấy class weights gây over-predict lớp hiếm


In [ ]:
PROJECT_ROOT = Path(".").resolve()
MANIFEST_PATH = PROJECT_ROOT / "reports" / "suggested_runs" / "image_only_fix" / "image_only_hash_split_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "suggested_runs" / "notebook_densenet_from_scratch"
TEST_DIR = PROJECT_ROOT / "data" / "test_images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "bacterial_leaf_blight",
    "bacterial_leaf_streak",
    "bacterial_panicle_blight",
    "blast",
    "brown_spot",
    "downy_mildew",
    "normal",
    "tungro",
]
LABEL_TO_IDX = {label: idx for idx, label in enumerate(CLASSES)}
IDX_TO_LABEL = {idx: label for label, idx in LABEL_TO_IDX.items()}

IMG_SIZE = 224
BATCH_SIZE = 48
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 7
LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-4
WEIGHT_DECAY = 1e-4
SEED = 7307
USE_CLASS_WEIGHTS = False

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()
print("Manifest:", MANIFEST_PATH)
print("Output:", OUTPUT_DIR)


## 3. Load manifest image-only và kiểm tra split


In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
required_cols = {"image_id", "relative_path", "image_path", "primary_disease", "dhash", "split"}
missing = required_cols - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest thiếu cột: {missing}")

manifest = manifest[manifest["primary_disease"].isin(CLASSES)].copy()
manifest["target"] = manifest["primary_disease"].map(LABEL_TO_IDX)

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "validation"].reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
display(pd.crosstab(manifest["primary_disease"], manifest["split"]))

leakage = manifest.groupby("dhash")["split"].nunique()
print("dHash groups crossing train/validation:", int((leakage > 1).sum()))


## 4. Dataset và DataLoader tự viết

Phần này tương tự template ViT: tự định nghĩa class `Dataset`, transform train/validation, rồi tạo `DataLoader`.


In [ ]:
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(18),
    T.ColorJitter(brightness=0.16, contrast=0.16, saturation=0.10, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class RiceDiseaseDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = int(row["target"])
        return image, label, idx

train_dataset = RiceDiseaseDataset(train_df, train_transforms)
val_dataset = RiceDiseaseDataset(val_df, val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

images, labels, indices = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)


## 5. Visualize batch mẫu


In [ ]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = tensor.cpu() * std + mean
    return image.clamp(0, 1)

def show_batch(loader, n=8):
    images, labels, _ = next(iter(loader))
    n = min(n, len(images))
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.ravel()
    for i in range(n):
        img = denormalize(images[i]).permute(1, 2, 0).numpy()
        axes[i].imshow(img)
        axes[i].set_title(IDX_TO_LABEL[int(labels[i])], fontsize=9)
        axes[i].axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()

show_batch(train_loader)


## 6. Build DenseNet121 model

Pipeline này dùng DenseNet121 pretrained ImageNet. Phần training được viết rõ 2 pha:

- Pha 1: freeze backbone, train classifier head
- Pha 2: unfreeze `denseblock4` và `norm5`, fine-tune với learning rate nhỏ


In [ ]:
def build_densenet121(num_classes=len(CLASSES)):
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    return model

def set_trainable_phase(model, phase):
    for param in model.features.parameters():
        param.requires_grad = False

    if phase == 2:
        for name, param in model.features.named_parameters():
            if name.startswith("denseblock4") or name.startswith("norm5"):
                param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

def count_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

model = build_densenet121().to(device)
set_trainable_phase(model, phase=1)
total_params, trainable_params = count_trainable(model)
print(f"Total parameters: {total_params:,}")
print(f"Trainable phase 1: {trainable_params:,}")


## 7. Loss, optimizer, metrics


In [ ]:
def make_criterion(train_frame, use_class_weights=False):
    if not use_class_weights:
        return nn.CrossEntropyLoss()

    counts = train_frame["primary_disease"].value_counts().reindex(CLASSES, fill_value=0)
    weights = counts.sum() / (len(CLASSES) * counts.clip(lower=1))
    print("Class weights:")
    display(weights)
    return nn.CrossEntropyLoss(weight=torch.tensor(weights.values, dtype=torch.float32, device=device))

criterion = make_criterion(train_df, USE_CLASS_WEIGHTS)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


## 8. Training loop tự viết

Cell này hoàn chỉnh các hàm train/evaluate giống tinh thần template ViT, nhưng dùng model PyTorch `torchvision.models.densenet121`.


In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels, _ in tqdm(dataloader, desc="Train", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.item()) * labels.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    metrics = compute_metrics(all_labels, all_preds)
    return total_loss / len(dataloader.dataset), metrics

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs, all_indices = [], [], [], []

    with torch.no_grad():
        for images, labels, indices in tqdm(dataloader, desc="Eval", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)

            total_loss += float(loss.item()) * labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_indices.extend(indices.numpy().tolist())

    metrics = compute_metrics(all_labels, all_preds)
    return total_loss / len(dataloader.dataset), metrics, np.array(all_labels), np.array(all_preds), np.array(all_probs), all_indices


## 9. Chạy fine-tune 2 pha


In [ ]:
history = []
best = {
    "macro_f1": -1.0,
    "epoch": None,
    "path": OUTPUT_DIR / "best_densenet121_from_scratch_pipeline.pt",
}

global_epoch = 0
start_time = time.time()

for phase, num_epochs, lr in [
    (1, PHASE1_EPOCHS, LR_PHASE1),
    (2, PHASE2_EPOCHS, LR_PHASE2),
]:
    set_trainable_phase(model, phase)
    total_params, trainable_params = count_trainable(model)
    print(f"\nPhase {phase}: lr={lr}, trainable={trainable_params:,}/{total_params:,}")

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

    for _ in range(num_epochs):
        global_epoch += 1
        train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
        val_loss, val_metrics, y_true, y_pred, y_prob, row_indices = evaluate(model, val_loader, criterion, device)

        row = {
            "epoch": global_epoch,
            "phase": phase,
            "lr": lr,
            "train_loss": train_loss,
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_macro_recall": val_metrics["macro_recall"],
        }
        history.append(row)
        print(json.dumps(row, indent=2))

        if row["val_macro_f1"] > best["macro_f1"]:
            best.update({"macro_f1": row["val_macro_f1"], "epoch": global_epoch})
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "classes": CLASSES,
                    "epoch": global_epoch,
                    "history": history,
                    "config": {
                        "img_size": IMG_SIZE,
                        "batch_size": BATCH_SIZE,
                        "phase1_epochs": PHASE1_EPOCHS,
                        "phase2_epochs": PHASE2_EPOCHS,
                        "use_class_weights": USE_CLASS_WEIGHTS,
                    },
                },
                best["path"],
            )
            print(f"Saved best checkpoint: epoch={best['epoch']}, macro_f1={best['macro_f1']:.4f}")

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "train_log.csv", index=False)
print(f"Training done in {(time.time() - start_time) / 60:.2f} minutes")
print("Best:", best)


## 10. Visualize learning curves


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="s", label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="train acc")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], marker="s", label="val acc")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="val macro F1")
axes[2].plot(history_df["epoch"], history_df["val_macro_recall"], marker="s", label="val macro recall")
axes[2].set_title("Macro metrics")
axes[2].set_xlabel("Epoch")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=160)
plt.show()


## 11. Final evaluation và confusion matrix


In [ ]:
checkpoint = torch.load(best["path"], map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])

val_loss, val_metrics, y_true, y_pred, y_prob, row_indices = evaluate(model, val_loader, criterion, device)

print("Final validation loss:", val_loss)
print(json.dumps(val_metrics, indent=2))
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))

pred_df = val_df.iloc[row_indices].copy()
pred_df["y_true"] = [IDX_TO_LABEL[i] for i in y_true]
pred_df["y_pred"] = [IDX_TO_LABEL[i] for i in y_pred]
pred_df["confidence"] = y_prob.max(axis=1)
pred_df.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("DenseNet121 Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=160)
plt.show()


## 12. Error gallery


In [ ]:
def make_error_gallery(predictions, max_images=32):
    errors = predictions[predictions["y_true"] != predictions["y_pred"]].copy()
    if errors.empty:
        print("Không có lỗi trên validation.")
        return

    errors = errors.sort_values("confidence", ascending=False).head(max_images)
    cols = 4
    rows = int(np.ceil(len(errors) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, errors.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")
        ax.imshow(image)
        ax.set_title(
            f"T: {row['y_true']}\nP: {row['y_pred']} ({row['confidence']:.2f})",
            fontsize=8,
        )
        ax.axis("off")

    for ax in axes[len(errors):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "error_gallery.png", dpi=160)
    plt.show()

make_error_gallery(pred_df)


## 13. Inference cho một ảnh


In [ ]:
def predict_image(model, image_path, transform=val_transforms, threshold=0.50):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    confidence = float(probs[pred_idx])
    label = IDX_TO_LABEL[pred_idx]
    decision = label if confidence >= threshold else "needs_review"
    return decision, label, confidence, probs

sample_path = val_df.iloc[0]["image_path"]
decision, label, confidence, probs = predict_image(model, sample_path)
print("Image:", sample_path)
print("Decision:", decision)
print("Predicted label:", label)
print("Confidence:", confidence)


## 14. Tạo submission cho test_images


In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, image_dir: Path, transform=None):
        self.paths = sorted([p for p in image_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, path.name

def create_submission(model, test_dir=TEST_DIR, out_path=OUTPUT_DIR / "submission.csv"):
    if not test_dir.exists():
        print(f"Không tìm thấy test dir: {test_dir}")
        return None

    test_dataset = TestImageDataset(test_dir, val_transforms)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    rows = []
    model.eval()
    with torch.no_grad():
        for images, image_ids in tqdm(test_loader, desc="Predict test"):
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1).cpu().numpy().tolist()
            confs = probs.max(dim=1).values.cpu().numpy().tolist()
            for image_id, pred_idx, conf in zip(image_ids, preds, confs):
                rows.append({
                    "image_id": image_id,
                    "label": IDX_TO_LABEL[int(pred_idx)],
                    "confidence": float(conf),
                })

    submission = pd.DataFrame(rows)
    submission[["image_id", "label"]].to_csv(out_path, index=False)
    submission.to_csv(OUTPUT_DIR / "submission_with_confidence.csv", index=False)
    print("Saved:", out_path)
    return submission

submission = create_submission(model)
if submission is not None:
    display(submission.head())


## 15. Summary

Notebook này là pipeline DenseNet121 from-scratch theo phong cách template ViT:

- Không còn Kaggle token hard-code
- Không dùng TensorFlow/Keras Colab code cũ
- Không dùng metadata giả
- Có Dataset/DataLoader tự viết
- Có training loop/evaluation loop tự viết
- Có fine-tune 2 pha
- Có confusion matrix, error gallery, inference và submission

Kết quả tốt nhất từ script tương ứng hiện tại:

- `EX07_DENSENET_GPU_IMAGE_ONLY_FIXED`
- Accuracy `0.9051`
- Macro F1 `0.8984`
- Checkpoint: `reports/suggested_runs/image_only_fix/densenet_gpu_no_weights/best_densenet121_gpu_no_weights.pt`
